# 01 — Extracción de demanda eléctrica peninsular desde e·sios

TFM — Forecasting de demanda energética en España.

Este notebook convierte la extracción inicial de e·sios en un flujo reproducible y compatible con la estructura del resto de datasets del proyecto.

**Objetivo del notebook**

1. Descargar demanda eléctrica peninsular desde e·sios / Red Eléctrica.
2. Limpiar la respuesta de la API.
3. Generar un dataset horario con la variable objetivo `demanda_mw`.
4. Guardar salidas `raw` y `processed`.
5. Preparar el dataset para cruzarlo con temperatura nacional ponderada y otras variables explicativas.

**Idea clave**

La demanda peninsular no se pondera por población. Ya es una magnitud agregada a nivel peninsular.  
Lo que sí se pondera es la temperatura nacional, porque se calcula a partir de ciudades o zonas con distinto peso poblacional.

## 1. Configuración del proyecto

Estructura esperada:

```text
TFM/
├── 01_extraccion_demanda_esios_tfm.ipynb
├── fetch_esios_demanda.py
├── build_dataset_modelado.py
├── data/
│   ├── raw/
│   └── processed/
```

El notebook asume que `fetch_esios_demanda.py` está en la misma carpeta que el notebook.

In [ ]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
import numpy as np
import pyarrow

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_ROOT: d:\NICO\Documentos\Master Data Science\TFM\datos demanda
RAW_DIR: d:\NICO\Documentos\Master Data Science\TFM\datos demanda\data\raw
PROCESSED_DIR: d:\NICO\Documentos\Master Data Science\TFM\datos demanda\data\processed


## 2. Token de e·sios

Por seguridad, el token no debe quedar escrito dentro del notebook.

Opciones recomendadas:

### Opción A — Terminal

```bash
export ESIOS_TOKEN="tu_token"
```

### Opción B — Notebook, solo para ejecución local

Descomenta la línea siguiente si estás trabajando localmente.  
No subas el notebook a GitHub con el token escrito.

In [4]:
# SOLO para ejecución local.
# Descomentar la siguiente línea si no se usa variable de entorno.
os.environ["ESIOS_TOKEN"] = "627ea9fe02867d78de6783bb260e2b56ebe7d612542b59d2576112b891cc2279"

TOKEN = os.getenv("ESIOS_TOKEN")

if TOKEN is None:
    raise ValueError(
        "No se encontró ESIOS_TOKEN. "
        "Define la variable de entorno o descomenta la línea anterior."
    )

print("✅ Token ESIOS detectado.")

✅ Token ESIOS detectado.


## 3. Importar el extractor

El extractor contiene la lógica reutilizable:

- conexión con la API
- descarga por bloques
- limpieza
- filtrado peninsular
- validaciones básicas
- guardado en CSV y Parquet

In [5]:
import importlib
import fetch_esios_demanda

importlib.reload(fetch_esios_demanda)
from fetch_esios_demanda import EsiosDemandExtractor

## 4. Parámetros de extracción

Para el TFM se recomienda extraer al menos 5 años si el objetivo es modelar patrones:

- estacionalidad anual
- estacionalidad semanal
- efecto de temperatura
- festivos y anomalías
- lags de demanda

Ajusta `START_DATE` y `END_DATE` según el rango que uses en el resto de datasets.

In [6]:
INDICATOR_ID = 1293      # Demanda real
START_DATE = "2021-01-01"
END_DATE = None          # None = hasta hoy
CHUNK_DAYS = 30
GEO_FILTER = "Península"

config = {
    "indicator_id": INDICATOR_ID,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "chunk_days": CHUNK_DAYS,
    "geo_filter": GEO_FILTER,
}

config

{'indicator_id': 1293,
 'start_date': '2021-01-01',
 'end_date': None,
 'chunk_days': 30,
 'geo_filter': 'Península'}

## 5. Ejecutar extracción

Esta celda descarga los datos y genera dos datasets:

```text
data/raw/demanda_esios_raw.csv
data/raw/demanda_esios_raw.parquet

data/processed/demanda_peninsular_horaria.csv
data/processed/demanda_peninsular_horaria.parquet
```

In [ ]:
extractor = EsiosDemandExtractor.from_env(
    indicator_id=INDICATOR_ID,
    start_date=START_DATE,
    end_date=END_DATE,
    chunk_days=CHUNK_DAYS,
    geo_filter=GEO_FILTER,
)

outputs = extractor.run()

demanda_raw = outputs["demanda_esios_raw"]
demanda = outputs["demanda_peninsular_horaria"]

print("RAW:", demanda_raw.shape)
print("PROCESSED:", demanda.shape)

demanda.head(10)

13:27:29 [INFO] TFM — Ingesta demanda e·sios
13:27:29 [INFO] Indicador: 1293
13:27:29 [INFO] Ventana: 2021-01-01 → hoy
13:27:30 [INFO] Descargando e·sios: 2021-01-01 00:00:00 → 2021-01-31 00:00:00
13:27:51 [INFO] Descargando e·sios: 2021-01-31 00:00:00 → 2021-03-02 00:00:00
13:27:57 [INFO] Descargando e·sios: 2021-03-02 00:00:00 → 2021-04-01 00:00:00
13:28:00 [INFO] Descargando e·sios: 2021-04-01 00:00:00 → 2021-05-01 00:00:00
13:28:02 [INFO] Descargando e·sios: 2021-05-01 00:00:00 → 2021-05-31 00:00:00
13:28:05 [INFO] Descargando e·sios: 2021-05-31 00:00:00 → 2021-06-30 00:00:00
13:28:07 [INFO] Descargando e·sios: 2021-06-30 00:00:00 → 2021-07-30 00:00:00
13:28:12 [INFO] Descargando e·sios: 2021-07-30 00:00:00 → 2021-08-29 00:00:00
13:28:14 [INFO] Descargando e·sios: 2021-08-29 00:00:00 → 2021-09-28 00:00:00
13:28:17 [INFO] Descargando e·sios: 2021-09-28 00:00:00 → 2021-10-28 00:00:00
13:28:19 [INFO] Descargando e·sios: 2021-10-28 00:00:00 → 2021-11-27 00:00:00
13:28:22 [INFO] Descarg

RAW: (500783, 7)
PROCESSED: (500663, 15)


,datetime,datetime_utc,demanda_mw,geo_id,geo_name,tz_time,indicator_id,fecha,año,mes,dia,hora,dia_semana,es_fin_de_semana,fuente
0,2021-01-01 00:00:00,2020-12-31 23:00:00+00:00,25134.0,8741,Península,2020-12-31T23:00:00.000Z,1293,2021-01-01,2021,1,1,0,4,0,e·sios / Red Eléctrica
1,2021-01-01 00:10:00,2020-12-31 23:10:00+00:00,24907.0,8741,Península,2020-12-31T23:10:00.000Z,1293,2021-01-01,2021,1,1,0,4,0,e·sios / Red Eléctrica
2,2021-01-01 00:20:00,2020-12-31 23:20:00+00:00,24787.0,8741,Península,2020-12-31T23:20:00.000Z,1293,2021-01-01,2021,1,1,0,4,0,e·sios / Red Eléctrica
3,2021-01-01 00:30:00,2020-12-31 23:30:00+00:00,24600.0,8741,Península,2020-12-31T23:30:00.000Z,1293,2021-01-01,2021,1,1,0,4,0,e·sios / Red Eléctrica
4,2021-01-01 00:40:00,2020-12-31 23:40:00+00:00,24537.0,8741,Península,2020-12-31T23:40:00.000Z,1293,2021-01-01,2021,1,1,0,4,0,e·sios / Red Eléctrica


## 6. Validación de calidad

Antes de usar el dataset como variable objetivo, comprobamos:

- rango temporal
- duplicados por `datetime`
- horas faltantes
- nulos en `demanda_mw`
- rango mínimo/máximo de demanda

In [9]:
qa_report = extractor.quality_report(demanda)
qa_report

{'rows': 500663,
 'start': '2021-01-01 00:00:00',
 'end': '2026-06-16 13:20:00',
 'duplicated_datetime': 0,
 'missing_hours': 6,
 'null_demanda_mw': 0,
 'min_demanda_mw': 0.0,
 'max_demanda_mw': 42052.0,
 'mean_demanda_mw': 26971.753810447346}

In [10]:
assert demanda["datetime"].isna().sum() == 0, "Hay datetimes nulos"
assert demanda["demanda_mw"].isna().sum() == 0, "Hay demanda_mw nula"
assert demanda.duplicated("datetime").sum() == 0, "Hay datetimes duplicados"

print("✅ Validaciones básicas superadas")

✅ Validaciones básicas superadas


## 7. Inspección rápida

Esta parte no es imprescindible para el pipeline, pero ayuda a detectar errores groseros de extracción.

In [12]:
demanda[["datetime", "demanda_mw", "geo_name", "fecha", "hora"]].head(10)

,datetime,demanda_mw,geo_name,fecha,hora
0,2021-01-01 00:00:00,25134.0,Península,2021-01-01,0
1,2021-01-01 00:10:00,24907.0,Península,2021-01-01,0
2,2021-01-01 00:20:00,24787.0,Península,2021-01-01,0
3,2021-01-01 00:30:00,24600.0,Península,2021-01-01,0
4,2021-01-01 00:40:00,24537.0,Península,2021-01-01,0
5,2021-01-01 00:50:00,24372.0,Península,2021-01-01,0
6,2021-01-01 01:00:00,24179.0,Península,2021-01-01,1
7,2021-01-01 01:10:00,23945.0,Península,2021-01-01,1
8,2021-01-01 01:20:00,23703.0,Península,2021-01-01,1
9,2021-01-01 01:30:00,23505.0,Península,2021-01-01,1


In [13]:
demanda["demanda_mw"].describe()

count    500663.000000
mean      26971.753810
std        4401.039678
min           0.000000
25%       23380.000000
50%       26982.000000
75%       30240.000000
max       42052.000000
Name: demanda_mw, dtype: float64

In [14]:
# Serie diaria media para inspección rápida
demanda_diaria = (
    demanda
    .assign(fecha=pd.to_datetime(demanda["fecha"]))
    .groupby("fecha", as_index=False)["demanda_mw"]
    .mean()
)

demanda_diaria.head()

,fecha,demanda_mw
0,2021-01-01,23938.006944
1,2021-01-02,26860.361111
2,2021-01-03,27079.875000
3,2021-01-04,31693.784722
4,2021-01-05,32275.888889


## 8. Encaje con temperatura nacional ponderada

La demanda es horaria.  
La temperatura nacional ponderada generada por el pipeline meteorológico suele ser diaria.

Por tanto, el primer merge razonable es:

```text
demanda horaria many-to-one temperatura diaria
```

Clave de unión:

```text
fecha
```

Más adelante, si se genera temperatura horaria nacional ponderada, el merge debería hacerse por:

```text
datetime
```

In [16]:
temp_path_csv = PROCESSED_DIR / "temperatura_nacional_ponderada.csv"
temp_path_parquet = PROCESSED_DIR / "temperatura_nacional_ponderada.parquet"

if temp_path_parquet.exists():
    temp = pd.read_parquet(temp_path_parquet)
elif temp_path_csv.exists():
    temp = pd.read_csv(temp_path_csv)
else:
    temp = None
    print("⚠️ No se encontró temperatura_nacional_ponderada en data/processed")

if temp is not None:
    temp["fecha"] = pd.to_datetime(temp["fecha"]).dt.date.astype(str)
    demanda["fecha"] = pd.to_datetime(demanda["fecha"]).dt.date.astype(str)

    dataset_modelado_base = demanda.merge(
        temp,
        on="fecha",
        how="left",
        validate="many_to_one"
    )

    print(dataset_modelado_base.shape)
    display(dataset_modelado_base.head())

(500663, 25)


,datetime,datetime_utc,demanda_mw,geo_id,geo_name,tz_time,indicator_id,fecha,año_x,mes_x,...,t_nac_media,t_nac_max,t_nac_min,sensacion_nac,ciudades_usadas,HDD,CDD,año_y,mes_y,fuente_y
0,2021-01-01 00:00:00,2020-12-31 23:00:00+00:00,25134.0,8741,Península,2020-12-31T23:00:00.000Z,1293,2021-01-01,2021,1,...,3.819409,5.851205,2.272445,2.606283,3.0,14.180591,0.0,2021.0,1.0,Open-Meteo (ERA5) + INE Padrón
1,2021-01-01 00:10:00,2020-12-31 23:10:00+00:00,24907.0,8741,Península,2020-12-31T23:10:00.000Z,1293,2021-01-01,2021,1,...,3.819409,5.851205,2.272445,2.606283,3.0,14.180591,0.0,2021.0,1.0,Open-Meteo (ERA5) + INE Padrón
2,2021-01-01 00:20:00,2020-12-31 23:20:00+00:00,24787.0,8741,Península,2020-12-31T23:20:00.000Z,1293,2021-01-01,2021,1,...,3.819409,5.851205,2.272445,2.606283,3.0,14.180591,0.0,2021.0,1.0,Open-Meteo (ERA5) + INE Padrón
3,2021-01-01 00:30:00,2020-12-31 23:30:00+00:00,24600.0,8741,Península,2020-12-31T23:30:00.000Z,1293,2021-01-01,2021,1,...,3.819409,5.851205,2.272445,2.606283,3.0,14.180591,0.0,2021.0,1.0,Open-Meteo (ERA5) + INE Padrón
4,2021-01-01 00:40:00,2020-12-31 23:40:00+00:00,24537.0,8741,Península,2020-12-31T23:40:00.000Z,1293,2021-01-01,2021,1,...,3.819409,5.851205,2.272445,2.606283,3.0,14.180591,0.0,2021.0,1.0,Open-Meteo (ERA5) + INE Padrón


## 9. Features temporales y lags iniciales

Estas variables suelen ser útiles para forecasting de demanda:

- hora del día
- día de la semana
- fin de semana
- mes
- codificación cíclica de hora y mes
- demanda hace 24 horas
- demanda hace 168 horas, es decir, una semana

In [19]:
dataset_modelado_base = dataset_modelado_base.rename(columns={
    "año_x": "año",
    "mes_x": "mes",
    "fuente_x": "fuente_demanda",
    "fuente_y": "fuente_meteo"
})

In [20]:
dataset_modelado_base = dataset_modelado_base.drop(
    columns=["año_y", "mes_y"],
    errors="ignore"
)

In [21]:
def add_cyclical_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["hora_sin"] = np.sin(2 * np.pi * df["hora"] / 24)
    df["hora_cos"] = np.cos(2 * np.pi * df["hora"] / 24)
    df["mes_sin"] = np.sin(2 * np.pi * df["mes"] / 12)
    df["mes_cos"] = np.cos(2 * np.pi * df["mes"] / 12)
    return df

def add_lags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("datetime").copy()
    df["demanda_lag_24h"] = df["demanda_mw"].shift(24)
    df["demanda_lag_168h"] = df["demanda_mw"].shift(168)
    df["demanda_rolling_24h"] = df["demanda_mw"].shift(1).rolling(24).mean()
    df["demanda_rolling_168h"] = df["demanda_mw"].shift(1).rolling(168).mean()
    return df

if "dataset_modelado_base" in globals():
    dataset_modelado_base = add_cyclical_time_features(dataset_modelado_base)
    dataset_modelado_base = add_lags(dataset_modelado_base)

    output_path = PROCESSED_DIR / "dataset_modelado_base.parquet"
    dataset_modelado_base.to_parquet(output_path, index=False)
    dataset_modelado_base.to_csv(output_path.with_suffix(".csv"), index=False, encoding="utf-8-sig")

    print("Guardado:", output_path)
    display(dataset_modelado_base.head())

Guardado: d:\NICO\Documentos\Master Data Science\TFM\datos demanda\data\processed\dataset_modelado_base.parquet


,datetime,datetime_utc,demanda_mw,geo_id,geo_name,tz_time,indicator_id,fecha,año,mes,...,CDD,fuente_meteo,hora_sin,hora_cos,mes_sin,mes_cos,demanda_lag_24h,demanda_lag_168h,demanda_rolling_24h,demanda_rolling_168h
0,2021-01-01 00:00:00,2020-12-31 23:00:00+00:00,25134.0,8741,Península,2020-12-31T23:00:00.000Z,1293,2021-01-01,2021,1,...,0.0,Open-Meteo (ERA5) + INE Padrón,0.0,1.0,0.5,0.866025,NaN,NaN,NaN,NaN
1,2021-01-01 00:10:00,2020-12-31 23:10:00+00:00,24907.0,8741,Península,2020-12-31T23:10:00.000Z,1293,2021-01-01,2021,1,...,0.0,Open-Meteo (ERA5) + INE Padrón,0.0,1.0,0.5,0.866025,NaN,NaN,NaN,NaN
2,2021-01-01 00:20:00,2020-12-31 23:20:00+00:00,24787.0,8741,Península,2020-12-31T23:20:00.000Z,1293,2021-01-01,2021,1,...,0.0,Open-Meteo (ERA5) + INE Padrón,0.0,1.0,0.5,0.866025,NaN,NaN,NaN,NaN
3,2021-01-01 00:30:00,2020-12-31 23:30:00+00:00,24600.0,8741,Península,2020-12-31T23:30:00.000Z,1293,2021-01-01,2021,1,...,0.0,Open-Meteo (ERA5) + INE Padrón,0.0,1.0,0.5,0.866025,NaN,NaN,NaN,NaN
4,2021-01-01 00:40:00,2020-12-31 23:40:00+00:00,24537.0,8741,Península,2020-12-31T23:40:00.000Z,1293,2021-01-01,2021,1,...,0.0,Open-Meteo (ERA5) + INE Padrón,0.0,1.0,0.5,0.866025,NaN,NaN,NaN,NaN


In [ ]:
#Se detectan 6 horas ausentes, correspondientes al cambio oficial a horario de verano en España. No se imputan porque no representan pérdidas de datos, sino horas locales inexistentes.
df = demanda.copy()

df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values("datetime")

expected_range = pd.date_range(
    start=df["datetime"].min().floor("h"),
    end=df["datetime"].max().floor("h"),
    freq="h"
)

missing_hours = expected_range.difference(df["datetime"])

missing_hours 

DatetimeIndex(['2021-03-28 02:00:00', '2022-03-27 02:00:00',
               '2023-03-26 02:00:00', '2024-03-31 02:00:00',
               '2025-03-30 02:00:00', '2026-03-29 02:00:00'],
              dtype='datetime64[us]', freq=None)

In [23]:
missing_hours_df = pd.DataFrame({
    "missing_datetime": missing_hours
})

missing_hours_df["año"] = missing_hours_df["missing_datetime"].dt.year
missing_hours_df["mes"] = missing_hours_df["missing_datetime"].dt.month
missing_hours_df["dia"] = missing_hours_df["missing_datetime"].dt.day
missing_hours_df["hora"] = missing_hours_df["missing_datetime"].dt.hour
missing_hours_df["dia_semana"] = missing_hours_df["missing_datetime"].dt.day_name()

missing_hours_df

,missing_datetime,año,mes,dia,hora,dia_semana
0,2021-03-28 02:00:00,2021,3,28,2,Sunday
1,2022-03-27 02:00:00,2022,3,27,2,Sunday
2,2023-03-26 02:00:00,2023,3,26,2,Sunday
3,2024-03-31 02:00:00,2024,3,31,2,Sunday
4,2025-03-30 02:00:00,2025,3,30,2,Sunday
5,2026-03-29 02:00:00,2026,3,29,2,Sunday


In [ ]:
# Al hacer un checkeo de las horas, se detectaron 6 horas faltantes en la serie horaria local. 
# Estas horas coinciden con los cambios oficiales al horario de verano en España, donde la hora local 02:00 no existe por el salto horario a las 03:00. Por tanto, no se consideran valores perdidos reales ni se imputan.
expected_dst_missing = pd.to_datetime([
    "2021-03-28 02:00:00",
    "2022-03-27 02:00:00",
    "2023-03-26 02:00:00",
    "2024-03-31 02:00:00",
    "2025-03-30 02:00:00",
    "2026-03-29 02:00:00",
])

is_dst_gap = set(missing_hours) == set(expected_dst_missing)

print("Horas faltantes:", len(missing_hours))
print("Coinciden con el cambio de horario de verano:", is_dst_gap)

Horas faltantes: 6
Coinciden con el cambio de horario de verano: True


## 10. Dataset resultante esperado

Dataset principal para empezar análisis y modelado:

```text
data/processed/dataset_modelado_base.parquet
```

Columnas mínimas esperadas:

```text
datetime
datetime_utc
demanda_mw
geo_id
geo_name
tz_time
indicator_id
fecha
año
mes
dia
hora
dia_semana
es_fin_de_semana
fuente_demanda
t_nac_media
t_nac_max
t_nac_min
sensacion_nac
ciudades_usadas
HDD
CDD
fuente_meteo
hora_sin
hora_cos
mes_sin
mes_cos
demanda_lag_24h
demanda_lag_168h
demanda_rolling_24h
demanda_rolling_168h 
datetime

```

La variable objetivo es:

```text
demanda_mw
```

## 11. Resumen conceptual

### Concepto clave

Este notebook genera la variable objetivo del TFM: demanda eléctrica peninsular horaria.

### Regla práctica

- Datos de demanda: se agregan por sistema eléctrico, no por población.
- Datos meteorológicos: se ponderan por población para aproximar la exposición térmica nacional.
- Dataset de modelado: debe tener una fila por hora.
